# Lab 4.1 – PandasAI: fragen, prüfen, verbessern

**Starter.** Kopie in `project/playground/` bearbeiten.
Zellen der Reihe nach mit **Umschalt+Enter** ausführen. Fehlt der API-Schlüssel oder ist der Dienst gestört, zeigen die Fragezellen nur einen Hinweis; dann die Zelle **Ohne Dienst** ausführen.

| Teil | Inhalt |
| --- | --- |
| Setup | Tabelle laden, Sprachmodell einstellen, Hilfsfunktion `frage()` |
| Arbeitsphase 1 | vier vorgegebene Fragen, darunter eine Diagrammfrage; Gegenprobe mit pandas |
| Arbeitsphase 2 | Vorher/Nachher: Marge je Region, Prüfzelle gegen `data/kontrollsummen.csv` |

In [ ]:
import os

import pandas as pd
import pandasai as pai
from IPython.display import Image, display
from pandasai_litellm.litellm import LiteLLM

MODELL = "gpt-4.1-mini"     # Kursmodell, dasselbe wie in check_setup.py
EINGABE = "project/examples/m02-data-preparation/l03-features/umsatz_features.csv"
SOLLWERTE = "data/kontrollsummen.csv"

df = pai.read_csv(EINGABE)                           # pandas-Tabelle mit zusätzlicher Methode .chat()
MIT_DIENST = bool(os.environ.get("OPENAI_API_KEY"))  # Schlüssel kommt aus der Umgebungsvariable
if MIT_DIENST:
    pai.config.set({"llm": LiteLLM(model=MODELL, temperature=0)})   # temperature=0: möglichst gleiche Antworten
    print(f"Bereit: {len(df)} Zeilen, Modell {MODELL}")
else:
    print(f"{len(df)} Zeilen geladen. Ohne Dienst: OPENAI_API_KEY ist nicht gesetzt.")
    print("Die Fragezellen zeigen nur einen Hinweis; weiter mit der Zelle 'Ohne Dienst'.")

In [ ]:
def frage(text):
    """Stellt eine Frage an PandasAI, zeigt den erzeugten Code und die Antwort."""
    if not MIT_DIENST:
        print("Ohne Dienst – keine PandasAI-Antwort für:", text)
        return None
    try:
        antwort = df.chat(text)
    except Exception as fehler:                       # Netz, Schlüssel, Kontingent …
        print("Dienst nicht erreichbar:", type(fehler).__name__, "– weiter mit der Zelle 'Ohne Dienst'.")
        return None
    print("Erzeugter Code (mit den drei Lesefragen prüfen):")
    print(antwort.last_code_executed)
    print("\nAntwort:")
    if antwort.type == "chart":
        display(Image(filename=antwort.value))       # Diagramm liegt als PNG in exports/charts/
    else:
        display(antwort.value)                       # Zahl, Text oder Tabelle
    return antwort

## Testfrage

In [ ]:
test = frage("Wie viele Zeilen hat die Tabelle?")

## Arbeitsphase 1: vier Fragen, je Antwort den Code ansehen

In [ ]:
a1 = frage("Wie hoch war der Umsatz im September 2026 insgesamt?")

In [ ]:
a2 = frage("Welche Produktgruppe hatte 2025 den höchsten Umsatz?")

In [ ]:
a3 = frage("Wie viele verschiedene Kunden haben 2026 im Onlineshop bestellt?")

In [ ]:
a4 = frage("Zeichne den Umsatz je Monat als Liniendiagramm.")

### Gegenprobe mit pandas (fertig)

Dieselben Fragen, mit pandas gerechnet. Stimmt eine Antwort nicht, im erzeugten Code nach Filter, Spalte und Rechenart suchen.

In [ ]:
sep = df[df["monat"] == "2026-09"]
print("Umsatz September 2026:          ", round(sep["umsatz"].sum(), 2), "€")
print("Höchster Umsatz 2025:           ", df[df["jahr"] == 2025].groupby("produktgruppe")["umsatz"].sum().idxmax())
print("Kunden 2026 im Onlineshop:      ", df[(df["jahr"] == 2026) & (df["kanal"] == "Onlineshop")]["kunde_id"].nunique())

## Arbeitsphase 2: Vorher/Nachher bei der Marge

Sollwert: Marge je Region im September 2026 aus `data/kontrollsummen.csv` (Summe Deckungsbeitrag / Summe Umsatz).
Die Prüfhilfe trägt jeden Durchgang als Zeile in die Vergleichstabelle ein.

In [ ]:
# Prüfhilfe – fertig, nicht ändern
soll = pd.read_csv(SOLLWERTE)
soll_sep = soll[soll["monat"] == "2026-09"].set_index("region")["marge"]
vergleich = []


def marge_je_region(antwort):
    """Holt aus einer Antwort (Tabelle oder Reihe) die Marge je Region als Anteil, z. B. 0.2041."""
    wert = getattr(antwort, "value", antwort)
    if isinstance(wert, pd.Series):
        wert = wert.reset_index()
    if not isinstance(wert, pd.DataFrame):
        return None
    regionsspalten = [s for s in wert.columns if wert[s].astype(str).isin(soll_sep.index).any()]
    zahlenspalten = list(wert.select_dtypes("number").columns)
    margenspalten = [s for s in zahlenspalten if "marg" in str(s).lower()] or zahlenspalten[-1:]
    if not regionsspalten or not margenspalten:
        return None
    reihe = wert.set_index(regionsspalten[0])[margenspalten[0]].astype(float)
    return reihe / 100 if reihe.abs().max() > 1 else reihe     # Angabe in Prozent umrechnen


def pruefe_marge(antwort, durchgang):
    ist = marge_je_region(antwort)
    zeile = {"Durchgang": durchgang}
    if ist is None:
        zeile["Ergebnis"] = "nicht OK – keine Tabelle mit Region und Marge"
    else:
        ist = ist.reindex(soll_sep.index)
        for region in soll_sep.index:
            zeile[region] = round(ist[region] * 100, 2)
        abstand = (ist - soll_sep).abs().max()
        zeile["Ergebnis"] = "OK" if abstand <= 0.0005 else "nicht OK – bis zu " + f"{abstand * 100:.1f}".replace(".", ",") + " Prozentpunkte daneben"
    vergleich.append(zeile)
    print(zeile["Ergebnis"])
    sollzeile = {"Durchgang": "Sollwert (kontrollsummen.csv)", **(soll_sep * 100).round(2).to_dict(), "Ergebnis": ""}
    display(pd.DataFrame([sollzeile] + vergleich).set_index("Durchgang"))

### Vorher: ohne Spaltenbeschreibungen

In [ ]:
vorher = frage("Wie hoch war die Marge im September 2026 je Region? Gib eine Tabelle mit region und marge aus.")

In [ ]:
# Prüfzelle – nicht ändern
pruefe_marge(vorher, "Vorher: PandasAI ohne Beschreibungen")

### Spaltenbeschreibungen einsetzen (fertig)

Die Beschreibungen gehen mit jeder Frage an das Modell, genau wie Spaltennamen und die ersten fünf Zeilen.

In [ ]:
BESCHREIBUNGEN = {
    "umsatz": "Nettoumsatz der Buchung in Euro",
    "kosten": "Wareneinsatz der Buchung in Euro; leer, wenn die Kosten fehlen",
    "deckungsbeitrag": "umsatz minus kosten in Euro; leer, wenn die Kosten fehlen",
    "marge": "Marge EINER Buchung (deckungsbeitrag / umsatz). Nie mitteln: Die Marge einer Gruppe "
             "oder eines Monats ist SUM(deckungsbeitrag) / SUM(umsatz).",
    "monat": "Buchungsmonat als Text im Format JJJJ-MM, z. B. 2026-09",
    "region": "Verkaufsregion: Nord, Süd, West oder Ost",
}
for spalte in df.schema.columns:
    if spalte.name in BESCHREIBUNGEN:
        spalte.description = BESCHREIBUNGEN[spalte.name]
df.schema.description = "Umsatzbuchungen von Nordwind Handel, 01/2024 bis 09/2026, eine Zeile je Buchung"

print("Beschreibungen gesetzt für:", ", ".join(s.name for s in df.schema.columns if s.description))

### Nachher: mit Spaltenbeschreibungen, dieselbe Frage

In [ ]:
nachher = frage("Wie hoch war die Marge im September 2026 je Region? Gib eine Tabelle mit region und marge aus.")

In [ ]:
# Prüfzelle – nicht ändern
pruefe_marge(nachher, "Nachher: PandasAI mit Beschreibungen")

## Ohne Dienst: zwei pandas-Rechnungen zum Vergleich

Nur ausführen, wenn die Fragezellen „Ohne Dienst“ oder „Dienst nicht erreichbar“ melden. **Das ist keine PandasAI-Antwort**, sondern eine fertige pandas-Rechnung: links der typische Fehler (Mittelwert der Zeilenmargen), rechts die Regel aus Lab 2.3.

In [ ]:
sep = df[df["monat"] == "2026-09"]
mittelwert_der_zeilen = sep.groupby("region")["marge"].mean()            # typischer Fehler
summen = sep.groupby("region")[["umsatz", "deckungsbeitrag"]].sum()
aus_summen = summen["deckungsbeitrag"] / summen["umsatz"]                # Summe DB / Summe Umsatz

pruefe_marge(mittelwert_der_zeilen, "pandas: Mittelwert der Zeilenmargen (typischer Fehler)")
pruefe_marge(aus_summen, "pandas: Summe DB / Summe Umsatz (Regel)")

## Erweiterung: eigene Frage

Eigene Frage in Alltagssprache stellen und den erzeugten Code mit den drei Lesefragen prüfen: Welche Datei? Was wird verändert? Was wird ausgegeben?

In [ ]:
# eigene = frage("…")